# SmartPrice AI — modelagem temporal

Notebook documental. A lógica reutilizável vive em `src/`. O notebook 01 e artefatos V1 permanecem preservados. Execute `python -m src.train` para reproduzir a comparação completa; este notebook lê e confere os resultados salvos.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.config import *
from src.data_processing import load_modeling_data, validate_dataset, temporal_windows
from src.evaluation import regression_metrics
from src.inference import load_bundle, estimate_price
metadata = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
data = load_modeling_data()
validate_dataset(data)

{'rows': 63164, 'start': '2025-01-02', 'end': '2025-12-31', 'dtypes': {'Área Construída (m2)': 'int64', 'CEP_8': 'string', 'Padrão (IPTU)': 'string', 'ACC (IPTU)': 'int64', 'Valor de Transação (declarado pelo contribuinte)': 'float64', 'Data de Transação': 'datetime64[us]'}, 'missing': {'Área Construída (m2)': 0, 'CEP_8': 0, 'Padrão (IPTU)': 0, 'ACC (IPTU)': 0, 'Valor de Transação (declarado pelo contribuinte)': 0, 'Data de Transação': 0}, 'duplicate_technical_ids': 0, 'duplicate_reduced_rows': 750, 'duplicate_policy': 'Não remover: projeção reduzida não identifica duplicatas de transações. V3 removeu cópias exatas usando todas as colunas originais.', 'area_kind': 'Área Construída (m2) cadastral; não equivale necessariamente à área útil.', 'area': {'count': 63164.0, 'mean': 126.89875562029003, 'std': 98.0234898151958, 'min': 6.0, '25%': 71.0, '50%': 102.0, '75%': 148.0, 'max': 2023.0}, 'price': {'count': 63164.0, 'mean': 756163.2874145083, 'std': 1354584.9787531653, 'min': 10000.0, '25

## Dataset congelado e EDA
63.164 registros, 02/01–31/12/2025. Área construída cadastral; preço declarado, não necessariamente preço de mercado. Apenas quatro atributos cadastrais; sem proxies monetárias no modelo.

In [2]:
data[[AREA, TARGET]].describe(percentiles=[.01,.5,.95,.99])

       Área Construída (m2)  Valor de Transação (declarado pelo contribuinte)
count          63164.000000                                      6.316400e+04
mean             126.898756                                      7.561633e+05
std               98.023490                                      1.354585e+06
min                6.000000                                      1.000000e+04
1%                32.000000                                      7.817451e+04
50%              102.000000                                      4.480000e+05
95%              305.000000                                      2.100000e+06
99%              523.370000                                      5.150000e+06
max             2023.000000                                      7.000000e+07

In [3]:
data.assign(preco_m2=data[TARGET]/data[AREA]).groupby(data[DATE].dt.to_period('M')).agg(n=(TARGET,'size'),preco_mediano=(TARGET,'median'),m2_mediano=('preco_m2','median'))

                      n  preco_mediano   m2_mediano
Data de Transação                                  
2025-01            4512       460000.0  4797.775324
2025-02            5005       442444.0  4798.076923
2025-03            4893       450000.0  4859.154930
2025-04            5384       450000.0  4896.075321
2025-05            5653       450000.0  4950.495050
2025-06            5271       425000.0  4897.959184
2025-07            5329       440000.0  4935.483871
2025-08            5367       450000.0  4909.303043
2025-09            5513       442000.0  4900.000000
2025-10            5876       450000.0  5000.000000
2025-11            4897       450000.0  4975.124378
2025-12            5464       430000.0  4902.903469

## Features e split
Ridge preserva features e encoder V1; árvores acrescentam log de área e agrupamento postal de três dígitos. Data somente para split/EDA. CEP inédito aceito. Mesma representação em original/log dentro de cada família.

| Janela | Treino expansivo | n treino | Validação | n validação |
|---|---|---:|---|---:|
| 1 | 2025-01-02 a 2025-05-31 | 25447 | 2025-06-01 a 2025-07-31 | 10600 |
| 2 | 2025-01-02 a 2025-07-31 | 36047 | 2025-08-01 a 2025-08-31 | 5367 |
| 3 | 2025-01-02 a 2025-08-31 | 41414 | 2025-09-01 a 2025-10-21 | 9184 |


In [4]:
assert set(FEATURES).isdisjoint({TARGET, 'Preco_m2', DATE})
for train, valid in temporal_windows(data):
    assert train[DATE].max() < valid[DATE].min()
    assert valid[DATE].max() < pd.Timestamp(HOLDOUT_START)
metadata['split']

{'development': {'n': 50598, 'start': '2025-01-02', 'end': '2025-10-21'}, 'holdout': {'n': 12566, 'start': '2025-10-22', 'end': '2025-12-31'}, 'windows': [{'train': {'n': 25447, 'start': '2025-01-02', 'end': '2025-05-31'}, 'validation': {'n': 10600, 'start': '2025-06-01', 'end': '2025-07-31'}}, {'train': {'n': 36047, 'start': '2025-01-02', 'end': '2025-07-31'}, 'validation': {'n': 5367, 'start': '2025-08-01', 'end': '2025-08-31'}}, {'train': {'n': 41414, 'start': '2025-01-02', 'end': '2025-08-31'}, 'validation': {'n': 9184, 'start': '2025-09-01', 'end': '2025-10-21'}}]}

## Baselines, candidatos e validação
Mediana global, mediana de preço/m² por CEP treinada sem holdout, Ridge V1 e candidatos RF/HGB original/log. Escolha por MAE ponderado; estatísticas em reais após expm1.

| Modelo | MAE validação ponderado (R$) | Desvio entre janelas (R$) |
|---|---:|---:|
| hist_log | 193.496,89 | 4.266,15 |
| forest | 194.383,28 | 1.521,29 |
| forest_log | 195.700,41 | 2.377,80 |
| hist | 196.961,34 | 4.271,64 |
| cep_m2 | 234.181,61 | 5.717,93 |
| ridge_v1 | 279.928,42 | 9.562,30 |
| ridge_log | 337.584,43 | 48.817,15 |
| median | 461.851,78 | 11.083,67 |


In [5]:
pd.DataFrame([{'model':r['model'], **r['metrics'], 'fold_mae_std':r['fold_mae_std']} for r in metadata['selection']['ranking']])

        model      n            MAE          RMSE         R2          MedAE  MdAPE_pct     mean_error  underestimate_pct  overestimate_pct  negative_predictions  fold_mae_std
0    hist_log  25151  193496.889614  6.728360e+05   0.731772   78279.349651  17.548930  -49500.397684          56.327780         43.672220                     0   4266.148103
1      forest  25151  194383.282252  6.250310e+05   0.768533   83112.172445  17.751820  -10446.588032          48.403642         51.596358                     0   1521.286333
2  forest_log  25151  195700.414098  6.655321e+05   0.737564   81387.728690  17.955125  -62096.911812          56.113077         43.886923                     0   2377.798890
3        hist  25151  196961.344030  6.773006e+05   0.728200   81075.875227  17.652967   -5947.597120          49.365830         50.634170                     0   4271.644367
4      cep_m2  25151  234181.613658  8.761888e+05   0.545136   91025.641026  18.890430  -58592.796291          48.316170     

## Seleção e holdout
Vencedor: hist_log. Reajuste até 21/10/2025; holdout 22/10–31/12/2025. O holdout já havia sido examinado no diagnóstico histórico: não é teste prospectivo totalmente cego.

In [6]:
holdout = data.loc[data[DATE].ge(HOLDOUT_START)]
model, saved = load_bundle()
predicted = model.predict(holdout[FEATURES])
computed = regression_metrics(holdout[TARGET], predicted)
assert computed == saved['final_holdout_metrics']
computed

{'n': 12566, 'MAE': 212458.86298170176, 'RMSE': 810830.2003822536, 'R2': 0.7157420032469094, 'MedAE': 82655.00315080895, 'MdAPE_pct': 18.114918659918324, 'mean_error': -37013.556835933814, 'underestimate_pct': 54.26547827470954, 'overestimate_pct': 45.73452172529046, 'negative_predictions': 0}

In [7]:
pd.DataFrame(metadata['holdout_comparison']).T

                  n            MAE          RMSE           R2          MedAE  MdAPE_pct     mean_error  underestimate_pct  overestimate_pct  negative_predictions
median      12566.0  498921.904309  1.556114e+06    -0.046973  200000.000000  50.000000 -329608.966437          48.408404         50.740092                   0.0
cep_m2      12566.0  241802.550070  9.436990e+05     0.614948   86924.018603  18.063936  -42345.561155          48.607353         50.986790                   0.0
ridge_v1    12566.0  288838.433655  9.541081e+05     0.606406  124050.789267  25.148626   -9513.367245          52.005411         47.994589                 326.0
ridge_log   12566.0  840653.079613  5.443306e+07 -1280.086032   94696.574260  20.342135  539615.237737          53.437848         46.562152                   0.0
forest      12566.0  218097.523998  7.382323e+05     0.764365   91284.990694  18.856614   11040.676726          46.617858         53.382142                   0.0
forest_log  12566.0  215502.

## Cauda superior e categorias novas
A melhoria média não elimina erros na cauda. Não esconder negativos do Ridge.

| Faixa real | n | MAE (R$) | MedAE (R$) | Erro médio (R$) | Subestimação | Superestimação |
|---|---:|---:|---:|---:|---:|---:|
| Até R$ 500 mil | 7144 | 88.068,39 | 52.004,92 | 39.302,44 | 44,92% | 55,08% |
| R$ 500 mil–1 mi | 3105 | 162.520,83 | 121.584,89 | -13.673,81 | 63,80% | 36,20% |
| R$ 1–2 mi | 1563 | 338.581,34 | 273.552,20 | -106.870,32 | 70,44% | 29,56% |
| R$ 2–5 mi | 613 | 808.265,35 | 637.129,72 | -289.846,45 | 69,66% | 30,34% |
| Acima de R$ 5 mi | 141 | 3.626.248,59 | 2.124.778,15 | -2.544.096,60 | 71,63% | 28,37% |


In [8]:
metadata['prediction_policy'], metadata['unseen_categories']

({'nonnegative_clip': True, 'minimum': 0, 'selected_raw_negative_holdout': 0, 'selected_clipped_holdout': 0, 'ridge_original_negative_holdout': 326, 'explanation': 'Limite físico explícito >=0 somente na inferência final. Candidatos e Ridge histórico reportados sem clipping.'}, {'CEP_8': {'unique': 413, 'rows': 532}, 'Padrão (IPTU)': {'unique': 0, 'rows': 0}})

## Inferência e incerteza
Faixa baseada no erro histórico de validação, por preço previsto, com fallback para grupos pequenos. Não é intervalo de confiança ou garantia. Modelo experimental de ITBI de São Paulo; não substitui avaliação profissional.

In [9]:
estimate_price(70, '01234-000', '22', 2009)

{'prediction': 322553.95493870456, 'lower': 210398.14670868727, 'upper': 434709.7631687218, 'warnings': ['CEP inédito no treino: a estimativa usa a representação disponível e merece cautela adicional.']}

## Limitações e próximos passos
Uma coorte, área cadastral, valor declarado, poucas features, perda geográfica no prefixo postal e calibração compartilhando seleção. Validar prospectivamente antes de extrapolar para outras cidades/períodos. Leia README e reports/selection.json para metodologia e parâmetros completos.